# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EltunLTN/FlyRank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This notebook defines and verifies a small, honest contract for the Search Intelligence lane. Development uses March 2026, a mid-panel month, not the sealed June 2026 sample.


## 1. Contract — unit, tables, time, outcome, exclusion

**1. What one row means:** one row represents one content item for one client on one `report_date` in the daily performance fact table.

**2. Tables:** `fact_content_daily_performance` is the main table; `dim_clients` is used only to understand client history/availability and `dim_content` only for content context.

**3. Time window:** development is restricted to March 2026 (`report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'`). I use a mid-panel month so the final June 2026 month remains sealed.

**4. What I predict/rank:** a content item whose impressions decline by more than 20% month-over-month (`is_declining`), used as a proxy for search-performance risk.

**5. Deliberate exclusion:** `trend_pct` and `trend_direction` are excluded because they are derived from the outcome signal and would leak future/label information. IDs are also context only, never model features.


## 2. Fields — feature / label / context / excluded

**Features (safe at decision time):** `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, plus availability flags such as `ga4_data_available` when used only from the historical feature window.

**Label / proxy:** `is_declining`, derived from the future-vs-prior impression comparison.

**Context:** `client_hash_id`, `content_hash_id`, `report_date`, and client metadata. They identify, join, group, or split observations; they are not model inputs.

**Excluded:** `trend_pct`, `trend_direction`, and any field calculated from the outcome window. They are excluded to prevent target leakage.


## 3. Exactly three verification queries

The following three cells are the required checks. Run them in Colab after setting `HF_TOKEN`.


In [2]:
%pip -q install duckdb huggingface_hub
import os, duckdb
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('Set HF_TOKEN as a Colab Secret before running this notebook.')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
DAILY = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"


### Query 1 — grain
One `(report_date, client_hash_id, content_hash_id)` should identify at most one daily performance row.


In [3]:
# Verification query 1: grain
grain_check = con.sql(f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS rows_at_key
FROM {DAILY}
WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
GROUP BY 1,2,3
HAVING COUNT(*) > 1
LIMIT 5
""").df()
print('Duplicate grain keys:', len(grain_check))
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain keys: 0


,report_date,client_hash_id,content_hash_id,rows_at_key


### Query 2 — slice row count and date span
This measures the March 2026 development slice directly.


In [4]:
# Verification query 2: count + date span
slice_stats = con.sql(f"""
SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
FROM {DAILY}
WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
""").df()
slice_stats


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


### Query 3 — availability
GA4 rows are considered available only when `ga4_data_available IS TRUE`; zero-filled rows before a client's GA4 start are not treated as genuine zero engagement.


In [5]:
# Verification query 3: availability using IS TRUE
availability_stats = con.sql(f"""
SELECT
    COUNT(*) AS march_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS ga4_unavailable_rows
FROM {DAILY}
WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
""").df()
availability_stats


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_rows,ga4_available_rows,ga4_unavailable_rows
0,9841378,413966,9427412


## 3b. Five features

The feature frame below is deliberately small. It aggregates March observations per content/client. Each feature is historical information that could be known before the decision moment.

1. **`impressions_30d`** — available when the decision is made because it is calculated from the completed historical 30-day window before the prediction moment.

2. **`clicks_30d`** — available when the decision is made because clicks in the completed historical window have already been observed.

3. **`avg_position_30d`** — available when the decision is made because Search Console position measurements from the historical window are already recorded.

4. **`position_std_30d`** — available when the decision is made because it summarizes historical position variability only.

5. **`ga4_available_rate`** — available when the decision is made because it records the share of historical rows for which GA4 was actually available, using the explicit availability flag.


In [6]:
# Five-feature frame for March 2026.
features = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions_30d,
    SUM(gsc_clicks) AS clicks_30d,
    AVG(NULLIF(gsc_avg_position, 0)) AS avg_position_30d,
    STDDEV_SAMP(NULLIF(gsc_avg_position, 0)) AS position_std_30d,
    AVG(CASE WHEN ga4_data_available IS TRUE THEN 1.0 ELSE 0.0 END) AS ga4_available_rate
FROM {DAILY}
WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
GROUP BY 1,2
""").df()
print(f'Feature rows: {len(features):,}')
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows: 331,437


,client_hash_id,content_hash_id,impressions_30d,clicks_30d,avg_position_30d,position_std_30d,ga4_available_rate
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,181.0,0.0,5.331238,3.095192,0.0
1,client_62f4a7e64f5e0096,content_67741cce996cfafa,46.0,1.0,5.942308,2.940259,0.0
2,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,899.0,1.0,5.908100,5.911676,0.0
3,client_62f4a7e64f5e0096,content_ac8663da7484669a,34.0,0.0,6.419872,6.156393,0.0
4,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,3108.0,0.0,6.969536,2.286957,0.0


## 3c. Deliberate leakage trap

To demonstrate the trap, I intentionally add a label-derived column. The leaked feature is the target itself, so a quick classifier should approach a perfect score. This is not an acceptable model feature.


In [7]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Build a simple proxy label from the March slice: first half vs second half.
outcome = con.sql(f"""
WITH daily AS (
    SELECT client_hash_id, content_hash_id, report_date, SUM(gsc_impressions) AS impressions
    FROM {DAILY}
    WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
    GROUP BY 1,2,3
), agg AS (
    SELECT client_hash_id, content_hash_id,
           SUM(CASE WHEN report_date < DATE '2026-03-16' THEN impressions ELSE 0 END) AS prev_period,
           SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN impressions ELSE 0 END) AS outcome_period
    FROM daily GROUP BY 1,2
)
SELECT *, (outcome_period < 0.8 * prev_period)::INTEGER AS is_declining
FROM agg WHERE prev_period >= 100
""").df()

frame = features.merge(outcome[['client_hash_id','content_hash_id','is_declining']], on=['client_hash_id','content_hash_id'], how='inner').dropna()
X_train, X_test, y_train, y_test = train_test_split(frame[['impressions_30d']], frame['is_declining'], test_size=0.25, random_state=42, stratify=frame['is_declining'])

# Intentional leak: label-derived value is given directly to the model.
leaked = frame[['is_declining']].copy()
Xl_train, Xl_test, yl_train, yl_test = train_test_split(leaked, frame['is_declining'], test_size=0.25, random_state=42, stratify=frame['is_declining'])
leak_model = DecisionTreeClassifier(random_state=42).fit(Xl_train, yl_train)
leak_score = accuracy_score(yl_test, leak_model.predict(Xl_test))
print(f'Leaked quick score: {leak_score:.3f} — expected to be ~1.000 because the target itself was supplied.')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Leaked quick score: 1.000 — expected to be ~1.000 because the target itself was supplied.


In [8]:
# Remove the leaked column and keep the honest number.
honest_cols = ['impressions_30d', 'clicks_30d', 'avg_position_30d', 'position_std_30d', 'ga4_available_rate']
honest = frame.dropna(subset=honest_cols)
X_train, X_test, y_train, y_test = train_test_split(honest[honest_cols], honest['is_declining'], test_size=0.25, random_state=42, stratify=honest['is_declining'])
honest_model = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_train, y_train)
honest_score = accuracy_score(y_test, honest_model.predict(X_test))
print(f'Honest quick score after deleting leakage: {honest_score:.3f}')
print('Leakage feature retained in final feature list:', 'is_declining' in honest_cols)


Honest quick score after deleting leakage: 0.737
Leakage feature retained in final feature list: False


## 4. Limitation

**Named limitation — unbalanced history and availability:** clients do not all have the same history depth, and GA4 is explicitly unavailable for some rows. Therefore March observations are not equally informative across clients; an apparent signal may partly reflect which clients have usable history. The final model should use client-aware validation and availability filters rather than assuming a balanced panel.


## 5. Self-check

- [ ] Five plain-words contract answers are filled.
- [ ] Exactly three verification queries are present and their outputs are visible after Run All.
- [ ] Availability uses `IS TRUE`.
- [ ] Exactly five features are documented with an “available when?” explanation.
- [ ] The deliberate leakage experiment is shown, then the leaked column is removed.
- [ ] One limitation is named.
- [ ] No HF token is stored in the notebook.
- [ ] Run the notebook top to bottom in Colab, save the executed outputs, then commit the executed notebook.
